# ZPow rotation via a phase gradient

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## `ZPowConstViaPhaseGradient`
Apply an $Z^t$ on a qubit using a phase gradient state.

This bloq implements `Z**t` by conditionally loading `t/2` into a quantum
register, conditioned on the qubit `q` (rotation target), and then adding
this value to the phase gradient to get a phase kickback, and uncomputes the load.

See `PhaseGradientState` for details on phase gradients.

It loads an approximation of `t/2` to `phase_grad_bitsize` bits,
which uses `phase_grad_bitsize` clean ancilla. The total Tofolli cost
is `phase_grad_bitsize - 2`.

#### Parameters
 - `exponent`: value of `t` to apply `Z**t`
 - `phase_grad_bitsize`: number of qubits of the phase gradient state. 

#### Registers
 - `q`: qubit to apply rotation on.
 - `phase_grad`: phase gradient state of type `QFxp` with `phase_grad_bitsize` fractional bits. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix A: Addition for controlled rotations.
 - [Improved quantum circuits for elliptic curve discrete logarithms](https://arxiv.org/abs/2001.09580). Haner et al. 2020. Section 3: Components. "Integer addition" and Fig 2a describes the controlled-load trick.


In [ ]:
from examples.algorithms.zpow_phase_gradient_catalog import GradientZPowExample

### Example Instances

In [ ]:
zpow = GradientZPowExample.from_precision(3 / 8, epsilon=1e-11)

In [ ]:
symbolic_precision = GradientZPowExample.from_precision(3 / 8, epsilon=Symbol("eps"))

In [ ]:
symbolic_angle = GradientZPowExample.from_precision(Symbol("t"), epsilon=1e-11)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Concrete gradient ZPow", zpow.signature, {}),
    ("Symbolic precision", symbolic_precision.signature, {}),
    ("Symbolic angle", symbolic_angle.signature, {}),
])))

### Call Graph

In [ ]:
graph = zpow.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)

### Append the concrete operation

The caller supplies the gradient state as a through register. The body loads the quantized half-exponent under the data bit, adds it to the gradient and clears the loaded word. The measured carries also return to zero. The resource output separates measured lowering from the declared source model. Bind both symbolic variables to concrete values before appending an operation.

In [ ]:
circuit = DynamicCircuit(zpow.qubit_count)
circuit.append(zpow)
counts = Counter(gate.name for gate in circuit.to_ir().instructions)
print("Emitted on", zpow.qubit_count, "wires", dict(counts))
assert counts["measure"] == 39
for operation in (zpow, symbolic_precision, symbolic_angle):
    print(operation.resource_model())